# Benchmarking: all models

Every run recorded in [all_benchmark.json](../all_benchmark.json), collected in one place so all
numbers live in a single file. Per-venue subsets (with LaTeX output) live in
[satnac_2025/benchmark.ipynb](../satnac_2025/benchmark.ipynb) (torchvision models),
[sacair_2026/benchmark.ipynb](../sacair_2026/benchmark.ipynb) (slowfast MViTv2 models) and
[satnac_2026/benchmark.ipynb](../satnac_2026/benchmark.ipynb) (S3D, MViTv2_S, MViTv2_S_e).

Produced by [src/benchmark.py](../../benchmark.py). Environment details per run (GPU, driver,
library versions, git commit, clocks) are printed in the [Environment](#environment) section.

## Method

Read from `benchmark.py` (2026-09-28):

- **Isolation.** Each architecture runs in its own fresh subprocess, with launch order shuffled,
  so cuDNN caches/allocator state do not carry over and thermal drift is not tied to run order.
- **Input.** One real batch from the `asl100` train split (uniformly sampled to `num_frames`,
  random-cropped to `frame_size`=224, horizontal flip, model-specific normalisation). It is moved
  to the GPU once and reused for every iteration, so **timings exclude data loading and
  host-to-device transfer**.
- **Head.** 100 output classes, dropout 0.5.
- **Train (`full_step`).** `model.train()`; each iteration is `zero_grad` → forward →
  cross-entropy loss → backward → SGD step (lr 0.01, momentum 0.9).
- **Inference.** `model.eval()` + `torch.no_grad()` forward pass only.
- **Precision.** FP32 (no AMP), `torch.backends.cudnn.benchmark = True`.
- **Protocol.** Warm-up of 2 × 20 iterations, then 5 trials × 100 iterations. Each trial runs a
  *timing pass* (CUDA events around the 100 iterations) followed by a separate *monitoring pass*
  (NVML utilisation polled every 50 ms; peak memory from `torch.cuda.max_memory_allocated`), so
  the monitoring thread never perturbs the timings.
- **Metrics.** Latency = time per iteration, i.e. **per batch** (ms). Throughput = samples/s
  (= batches/s × batch size). All metrics are mean ± sample std over the 5 trials.
  Peak memory counts tensors allocated by PyTorch only — it excludes the CUDA context and the
  caching allocator's reserved-but-unused memory, so `nvidia-smi` would show more.
- **Batch-size sweep.** Batch sizes are tried in ascending powers of two per
  model/frame-count/mode; the first one that runs out of memory is recorded as `OOM` and the
  sweep stops, so the largest successful batch size is the maximum that fits in 12 GB.
- **Profiler cross-check.** After the trials, `torch.profiler` re-times 5 iterations as a sanity
  check on the CUDA-event latency (see [Profiler cross-check](#profiler-cross-check)).
- **Clocks.** Under load the GPU held a stable 1910–1930 MHz, so the SM clock was locked just
  below that with `sudo nvidia-smi -lgc 1900` (reset with `sudo nvidia-smi -rgc`).

## Models

- **torchvision** (Kinetics-400 pretrained): S3D, R3D_18, R(2+1)D_18, Swin3D_T/S/B, MViTv1_B,
  MViTv2_S, MViTv2_S_e.
- **slowfast reference implementation** (Kinetics-400 pretrained, see
  [src/models/README.md](../../models/README.md)): MViTv2_S_16x4, MViTv2_S_16x4_e,
  MViTv2_B_32x3, MViTv2_B_32x3_r.
- Suffix `_e` (*extended*): the 16-frame model run at 32 frames, with its temporal relative
  positional embeddings linearly interpolated to the longer input. Suffix `_r` (*reduced*): the
  32-frame MViTv2_B_32x3 run at 16 frames, interpolated the same way. MViTv2_B_32x3 and
  MViTv2_B_32x3_r are the same network, so the tables below show them as one model at 32 and
  16 frames respectively.

In [1]:
import json
from pathlib import Path
from typing import Literal, TypedDict

import pandas as pd
from typing_extensions import NotRequired

from src.run_types import RESULTS_DIR

## Load

`load_runs` flattens every entry into one row (OOM entries included, with empty metrics).

In [2]:
Mode = Literal["Train", "Infer"]


class MeanStd(TypedDict):
    mean: float
    std: float


class ProfilerComparison(TypedDict):
    profiler_latency_ms: float
    profiler_cpu_latency_ms: float
    iterations: int


class Results(TypedDict):
    n_trials: int
    latency_ms: MeanStd
    throughput_batches_per_s: MeanStd
    throughput_samples_per_s: MeanStd
    gpu_utilisation_percent: MeanStd
    gpu_utilisation_peak_percent: float
    peak_memory_mb: MeanStd
    raw_trial_latencies_ms: list[float]
    profiler_comparison: NotRequired[ProfilerComparison]


class Config(TypedDict):
    num_frames: int
    frame_size: int
    batch_size: int
    iterations: int
    trials: int
    full_step: NotRequired[bool]


class GpuClocks(TypedDict):
    sm_clock_mhz: int
    mem_clock_mhz: int
    sm_clock_max_mhz: int
    sm_clock_ratio_of_max: float


class Metadata(TypedDict):
    timestamp_utc: str
    git_commit: str
    git_dirty: bool
    torch_version: str
    cuda_version: str
    cudnn_version: int
    cudnn_benchmark_enabled: bool
    python_version: str
    gpu_name: str
    driver_version: str
    gpu_clocks: GpuClocks
    hostname: str


class GpuInfo(TypedDict):
    mem_total_mb: float


class Run(TypedDict):
    arch: str
    config: Config
    metadata: Metadata
    gpu_info: GpuInfo
    results: NotRequired[Results]
    error: NotRequired[str]


MODEL_ORDER: dict[str, str] = {
    "S3D": "S3D",
    "R3D_18": "R3D_18",
    "R(2+1)D_18": "R(2+1)D_18",
    "Swin3D_T": "Swin3D_T",
    "Swin3D_S": "Swin3D_S",
    "Swin3D_B": "Swin3D_B",
    "MViTv1_B": "MViTv1_B",
    "MViTv2_S": "MViTv2_S",
    "MViTv2_S_e": "MViTv2_S",
    "MViTv2_S_16x4": "MViTv2_S_16x4",
    "MViTv2_S_16x4_e": "MViTv2_S_16x4",
    "MViTv2_B_32x3_r": "MViTv2_B_32x3",
    "MViTv2_B_32x3": "MViTv2_B_32x3",
}
"""Arch name in the JSON -> base model it is a variant of, in display order."""


def run_row(run: Run) -> dict[str, object]:
    """Flatten one benchmark entry; metric columns are left out for OOM entries."""
    cfg, meta = run["config"], run["metadata"]
    row: dict[str, object] = {
        "arch": run["arch"],
        "base_model": MODEL_ORDER[run["arch"]],
        "mode": "Train" if cfg.get("full_step", False) else "Infer",
        "frames": cfg["num_frames"],
        "frame_size": cfg["frame_size"],
        "batch_size": cfg["batch_size"],
        "iterations": cfg["iterations"],
        "trials": cfg["trials"],
        "status": run.get("error", "ok"),
        "date_utc": meta["timestamp_utc"][:10],
        "git_commit": meta["git_commit"][:7],
        "git_dirty": meta["git_dirty"],
        "gpu_name": meta["gpu_name"],
        "gpu_mem_total_mb": run["gpu_info"]["mem_total_mb"],
        "driver": meta["driver_version"],
        "torch": meta["torch_version"],
        "cuda": meta["cuda_version"],
        "cudnn": meta["cudnn_version"],
        "cudnn_benchmark": meta["cudnn_benchmark_enabled"],
        "python": meta["python_version"],
        "sm_clock_mhz": meta["gpu_clocks"]["sm_clock_mhz"],
        "sm_clock_max_mhz": meta["gpu_clocks"]["sm_clock_max_mhz"],
        "mem_clock_mhz": meta["gpu_clocks"]["mem_clock_mhz"],
    }
    if "results" in run:
        res = run["results"]
        for key, col in (
            ("latency_ms", "latency_ms"),
            ("throughput_samples_per_s", "samples_per_s"),
            ("throughput_batches_per_s", "batches_per_s"),
            ("gpu_utilisation_percent", "gpu_util_pct"),
            ("peak_memory_mb", "peak_mem_mb"),
        ):
            row[f"{col}_mean"] = res[key]["mean"]
            row[f"{col}_std"] = res[key]["std"]
        row["gpu_util_peak_pct"] = res["gpu_utilisation_peak_percent"]
        row["raw_trial_latencies_ms"] = res["raw_trial_latencies_ms"]
        if "profiler_comparison" in res:
            prof = res["profiler_comparison"]
            row["profiler_latency_ms"] = prof["profiler_latency_ms"]
            row["profiler_cpu_latency_ms"] = prof["profiler_cpu_latency_ms"]
            row["profiler_iterations"] = prof["iterations"]
    return row


def load_runs(path: Path) -> pd.DataFrame:
    """Load every entry of an ``all_benchmark.json``-style file into one DataFrame.

    Rows are ordered by base model (``MODEL_ORDER``), mode, frame count and batch size. Raises
    ``KeyError`` on an arch missing from ``MODEL_ORDER`` so new models are not silently dropped.
    """
    with path.open() as f:
        runs: dict[str, Run] = json.load(f)["runs"]
    df = pd.DataFrame([run_row(run) for run in runs.values()])
    df["base_model"] = pd.Categorical(
        df["base_model"], categories=list(dict.fromkeys(MODEL_ORDER.values())), ordered=True
    )
    df["mode"] = pd.Categorical(df["mode"], categories=["Train", "Infer"], ordered=True)
    return df.sort_values(["base_model", "mode", "frames", "batch_size"]).reset_index(drop=True)


def show(df: pd.DataFrame, floatfmt: str = ".2f", numparse: bool = True) -> None:
    """Print as a markdown table, so the saved notebook output stays readable as plain text.

    Set ``numparse=False`` to print values verbatim (e.g. version strings like ``"12.4"``, which
    are otherwise reformatted as ``12.40``).
    """
    print(df.to_markdown(index=False, floatfmt=floatfmt, disable_numparse=not numparse))

In [3]:
runs_df = load_runs(RESULTS_DIR / "all_benchmark.json")
ok_df = runs_df[runs_df["status"] == "ok"]
print(
    f"{len(runs_df)} entries: {len(ok_df)} successful, "
    f"{(runs_df['status'] == 'OOM').sum()} OOM, "
    f"{runs_df['arch'].nunique()} arch names ({runs_df['base_model'].nunique()} base models)"
)

191 entries: 154 successful, 37 OOM, 13 arch names (10 base models)


## Environment

Distinct hardware/software configurations across all entries, with the arch names and dates
benchmarked under each. `git_dirty` means the working tree had uncommitted changes at run time.
Clocks are read once at the start of each run, before any load: the SM clock confirms the lock
held, but the memory clock was not locked, so its idle reading varies between runs.

In [4]:
env_cols = [
    "gpu_name", "gpu_mem_total_mb", "driver", "torch", "cuda", "cudnn", "cudnn_benchmark",
    "python", "sm_clock_mhz", "sm_clock_max_mhz", "mem_clock_mhz",
]
show(runs_df[env_cols].drop_duplicates(), numparse=False)
print()
provenance = (
    runs_df.groupby(["date_utc", "git_commit", "git_dirty"], observed=True)
    .agg(archs=("arch", lambda s: ", ".join(sorted(set(s)))), entries=("arch", "size"))
    .reset_index()
)
show(provenance)

| gpu_name                | gpu_mem_total_mb   | driver   | torch   | cuda   | cudnn   | cudnn_benchmark   | python   | sm_clock_mhz   | sm_clock_max_mhz   | mem_clock_mhz   |
|:------------------------|:-------------------|:---------|:--------|:-------|:--------|:------------------|:---------|:---------------|:-------------------|:----------------|
| NVIDIA GeForce RTX 3060 | 12288.0            | 595.84   | 2.5.1   | 12.4   | 90101   | True              | 3.10.18  | 1905           | 2100               | 7501            |
| NVIDIA GeForce RTX 3060 | 12288.0            | 595.84   | 2.5.1   | 12.4   | 90101   | True              | 3.10.18  | 1905           | 2100               | 810             |
| NVIDIA GeForce RTX 3060 | 12288.0            | 595.84   | 2.5.1   | 12.4   | 90101   | True              | 3.10.18  | 1905           | 2100               | 5001            |

| date_utc   | git_commit   | git_dirty   | archs                                                                     |

### Clock lock and `git_dirty` (checked 2026-10-06)

Both were questioned while reviewing the thesis. Neither affects the numbers:

- **The SM clock was locked for every run.** Idle and unlocked, this RTX 3060 reads 210 MHz SM /
  405 MHz memory (P8); idle with `nvidia-smi -lgc 1900` it reads 1905 / 810 MHz (P5). Every run
  above reads 1905 MHz at start, including those whose memory clock was still at its idle 810 MHz.
  `sm_clock_ratio_of_max` (0.91) is that locked clock as a fraction of the 2100 MHz boost maximum,
  not a sign of an unlocked card. Trial-to-trial latency spread is a median 0.02% (max 0.32%).
- **`git_dirty` is from the output file, not code.** `benchmark.py` writes into the tracked
  `all_benchmark.json`, so every run after the first in a batch saw a dirty tree. The commit made
  on the benchmark machine after each batch is a child of the recorded `git_commit` and changes
  only results: `bd50008` (after the 2026-09-12 runs on `c8a1748`: `all_benchmark.json`, one
  notebook) and `0e5e0d9` (after the 2026-09-14 runs on `85fa23a`: `all_benchmark.json`, a TODO).
  The first run after `bd50008` was clean. Since 2026-10-06, `benchmark.py` leaves the output file
  out of `git_dirty`, lists the dirty paths in `git_dirty_paths`, and stores the SM/memory clock
  range under load in each entry's results.
- **The numbers reproduce.** S3D at 16 frames, rerun on 2026-10-06 with the clock locked
  (train batch 1/4, inference batch 1/8, current code and labels), was within 0.15% of the
  latencies above, with the same peak memory.

## Batch-size coverage

Per model/frame count/mode: the batch sizes that ran, the largest that fit, and the batch size
that ran out of memory (blank where the sweep ended without an OOM).

In [5]:
def coverage(df: pd.DataFrame) -> pd.DataFrame:
    # Batch sizes run, the largest that fit and the first OOM, per arch/mode/frames
    rows: list[dict[str, object]] = []
    for (base, arch, mode, frames), grp in df.groupby(
        ["base_model", "arch", "mode", "frames"], observed=True, sort=True
    ):
        ok = grp.loc[grp["status"] == "ok", "batch_size"].sort_values()
        oom = grp.loc[grp["status"] == "OOM", "batch_size"]
        rows.append({
            "model": base, "arch": arch, "mode": mode, "frames": frames,
            "batch_sizes_ok": ", ".join(str(b) for b in ok),
            "max_bs": ok.max(),
            "oom_bs": str(oom.min()) if len(oom) else "",
        })
    return pd.DataFrame(rows)


show(coverage(runs_df))

| model         | arch            | mode   |   frames | batch_sizes_ok         |   max_bs |   oom_bs |
|:--------------|:----------------|:-------|---------:|:-----------------------|---------:|---------:|
| S3D           | S3D             | Train  |       16 | 1, 2, 4, 8, 16         |       16 |       32 |
| S3D           | S3D             | Train  |       32 | 1, 2, 4, 8             |        8 |       16 |
| S3D           | S3D             | Infer  |       16 | 1, 2, 4, 8, 16, 32, 64 |       64 |          |
| S3D           | S3D             | Infer  |       32 | 1, 2, 4, 8, 16, 32     |       32 |       64 |
| R3D_18        | R3D_18          | Train  |       16 | 1, 2, 4, 8, 16         |       16 |       32 |
| R3D_18        | R3D_18          | Train  |       32 | 1, 2, 4, 8             |        8 |       16 |
| R3D_18        | R3D_18          | Infer  |       16 | 1, 2, 4, 8, 16, 32     |       32 |       64 |
| R3D_18        | R3D_18          | Infer  |       32 | 1, 2, 4, 8, 16   

## Headline summary

One row per model and frame count: batch-size-1 latency and memory, the best throughput over the
sweep (and the batch size it was reached at), and the largest batch size that fit.

In [6]:
def summarise(df: pd.DataFrame, mode: Mode) -> pd.DataFrame:
    # One row per arch/frames for the given mode
    sub = df[(df["mode"] == mode) & (df["status"] == "ok")]
    rows: list[dict[str, object]] = []
    for (base, arch, frames), grp in sub.groupby(["base_model", "arch", "frames"], observed=True):
        bs1 = grp[grp["batch_size"] == 1].iloc[0]
        best = grp.loc[grp["samples_per_s_mean"].idxmax()]
        largest = grp.loc[grp["batch_size"].idxmax()]
        rows.append({
            "model": base, "arch": arch, "frames": frames,
            "bs1_latency_ms": bs1["latency_ms_mean"],
            "bs1_samples_per_s": bs1["samples_per_s_mean"],
            "bs1_peak_mem_mb": bs1["peak_mem_mb_mean"],
            "best_samples_per_s": best["samples_per_s_mean"],
            "best_at_bs": best["batch_size"],
            "max_bs": largest["batch_size"],
            "peak_mem_at_max_bs_mb": largest["peak_mem_mb_mean"],
        })
    return pd.DataFrame(rows)


for mode in ("Train", "Infer"):
    print(f"### {mode}")
    show(summarise(runs_df, mode))
    print()

### Train
| model         | arch            |   frames |   bs1_latency_ms |   bs1_samples_per_s |   bs1_peak_mem_mb |   best_samples_per_s |   best_at_bs |   max_bs |   peak_mem_at_max_bs_mb |
|:--------------|:----------------|---------:|-----------------:|--------------------:|------------------:|---------------------:|-------------:|---------:|------------------------:|
| S3D           | S3D             |       16 |            31.46 |               31.79 |            546.84 |                37.96 |           16 |       16 |                 7568.14 |
| S3D           | S3D             |       32 |            57.87 |               17.28 |           1010.27 |                18.87 |            8 |        8 |                 7565.88 |
| R3D_18        | R3D_18          |       16 |           129.96 |                7.69 |           1084.69 |                 8.33 |           16 |       16 |                10834.67 |
| R3D_18        | R3D_18          |       32 |           249.15 |          

### Per-model comparison at a fixed batch size

Latency and memory at batch size 1 side by side for training and inference, per frame count,
plus the train/infer latency ratio.

In [7]:
bs1 = ok_df[ok_df["batch_size"] == 1].pivot_table(
    index=["base_model", "arch", "frames"],
    columns="mode",
    values=["latency_ms_mean", "peak_mem_mb_mean"],
    observed=True,
)
bs1.columns = [f"{mode.lower()}_{metric.removesuffix('_mean')}" for metric, mode in bs1.columns]
bs1["train_over_infer_latency"] = bs1["train_latency_ms"] / bs1["infer_latency_ms"]
bs1 = bs1.reset_index().rename(columns={"base_model": "model"})
for frames in sorted(bs1["frames"].unique()):
    print(f"### {frames} frames, batch size 1")
    show(bs1[bs1["frames"] == frames].sort_values("infer_latency_ms"))
    print()

### 16 frames, batch size 1
| model         | arch            |   frames |   train_latency_ms |   infer_latency_ms |   train_peak_mem_mb |   infer_peak_mem_mb |   train_over_infer_latency |
|:--------------|:----------------|---------:|-------------------:|-------------------:|--------------------:|--------------------:|---------------------------:|
| S3D           | S3D             |       16 |              31.46 |               9.86 |              546.84 |              163.19 |                       3.19 |
| R3D_18        | R3D_18          |       16 |             129.96 |              34.60 |             1084.69 |              401.13 |                       3.76 |
| MViTv1_B      | MViTv1_B        |       16 |             127.93 |              40.50 |             1648.17 |              363.52 |                       3.16 |
| Swin3D_T      | Swin3D_T        |       16 |             120.12 |              42.98 |             1607.67 |              486.60 |                       2.79 |


## Full results

Every successful run, mean ± std over trials. `batches/s` = 1000 / latency; `util_peak` is the
highest NVML utilisation sample over all trials.

In [8]:
def pm(mean: pd.Series, std: pd.Series, fmt: str = ",.2f") -> pd.Series:
    return pd.Series(
        [f"{m:{fmt}} ± {s:{fmt}}" for m, s in zip(mean, std, strict=True)], index=mean.index
    )


def full_table(df: pd.DataFrame, mode: Mode) -> pd.DataFrame:
    # Every successful run for one mode, metrics formatted as mean ± std
    sub = df[(df["mode"] == mode) & (df["status"] == "ok")]
    return pd.DataFrame({
        "model": sub["base_model"],
        "arch": sub["arch"],
        "frames": sub["frames"],
        "bs": sub["batch_size"],
        "latency_ms": pm(sub["latency_ms_mean"], sub["latency_ms_std"]),
        "samples/s": pm(sub["samples_per_s_mean"], sub["samples_per_s_std"]),
        "batches/s": pm(sub["batches_per_s_mean"], sub["batches_per_s_std"]),
        "peak_mem_mb": pm(sub["peak_mem_mb_mean"], sub["peak_mem_mb_std"]),
        "gpu_util_pct": pm(sub["gpu_util_pct_mean"], sub["gpu_util_pct_std"]),
        "util_peak": sub["gpu_util_peak_pct"],
    })


for mode in ("Train", "Infer"):
    print(f"### {mode}")
    show(full_table(runs_df, mode), floatfmt=".0f")
    print()

### Train
| model         | arch            |   frames |   bs | latency_ms      | samples/s    | batches/s    | peak_mem_mb      | gpu_util_pct   |   util_peak |
|:--------------|:----------------|---------:|-----:|:----------------|:-------------|:-------------|:-----------------|:---------------|------------:|
| S3D           | S3D             |       16 |    1 | 31.46 ± 0.00    | 31.79 ± 0.00 | 31.79 ± 0.00 | 546.84 ± 0.00    | 99.97 ± 0.07   |         100 |
| S3D           | S3D             |       16 |    2 | 58.22 ± 0.00    | 34.35 ± 0.00 | 17.18 ± 0.00 | 1,012.14 ± 0.00  | 99.97 ± 0.05   |         100 |
| S3D           | S3D             |       16 |    4 | 109.85 ± 0.01   | 36.41 ± 0.00 | 9.10 ± 0.00  | 1,945.76 ± 0.00  | 99.99 ± 0.02   |         100 |
| S3D           | S3D             |       16 |    8 | 214.41 ± 0.01   | 37.31 ± 0.00 | 4.66 ± 0.00  | 3,821.83 ± 0.00  | 100.00 ± 0.01  |         100 |
| S3D           | S3D             |       16 |   16 | 421.49 ± 0.02   | 37.96 

## Run-to-run stability

Coefficient of variation of latency across the 5 trials (std / mean) and the raw per-trial
latencies, for checking that no run was noisy.

In [9]:
stability = ok_df[["base_model", "arch", "mode", "frames", "batch_size"]].copy()
stability["latency_cv_pct"] = 100 * ok_df["latency_ms_std"] / ok_df["latency_ms_mean"]
stability["raw_trial_latencies_ms"] = ok_df["raw_trial_latencies_ms"].map(
    lambda xs: ", ".join(f"{x:.2f}" for x in xs)
)
print(
    f"latency CV over all runs: max {stability['latency_cv_pct'].max():.3f}%, "
    f"median {stability['latency_cv_pct'].median():.3f}%"
)
show(stability.rename(columns={"base_model": "model"}), floatfmt=".3f")

latency CV over all runs: max 0.321%, median 0.019%
| model         | arch            | mode   |   frames |   batch_size |   latency_cv_pct | raw_trial_latencies_ms                      |
|:--------------|:----------------|:-------|---------:|-------------:|-----------------:|:--------------------------------------------|
| S3D           | S3D             | Train  |       16 |            1 |            0.008 | 31.46, 31.46, 31.45, 31.46, 31.46           |
| S3D           | S3D             | Train  |       16 |            2 |            0.006 | 58.21, 58.22, 58.22, 58.21, 58.22           |
| S3D           | S3D             | Train  |       16 |            4 |            0.009 | 109.87, 109.85, 109.84, 109.84, 109.86      |
| S3D           | S3D             | Train  |       16 |            8 |            0.003 | 214.40, 214.41, 214.42, 214.41, 214.42      |
| S3D           | S3D             | Train  |       16 |           16 |            0.004 | 421.47, 421.49, 421.51, 421.50, 421.49    

## Profiler cross-check

`torch.profiler` CUDA time per iteration (over 5 iterations) vs. the CUDA-event latency. The
profiler adds its own overhead, so it is expected to read somewhat higher; a large gap flags a
run worth re-checking.

In [10]:
prof = ok_df.dropna(subset=["profiler_latency_ms"])
prof_table = pd.DataFrame({
    "model": prof["base_model"],
    "arch": prof["arch"],
    "mode": prof["mode"],
    "frames": prof["frames"],
    "bs": prof["batch_size"],
    "event_latency_ms": prof["latency_ms_mean"],
    "profiler_cuda_ms": prof["profiler_latency_ms"],
    "profiler_cpu_ms": prof["profiler_cpu_latency_ms"],
    "cuda_diff_pct": 100 * (prof["profiler_latency_ms"] / prof["latency_ms_mean"] - 1),
})
print(
    f"{len(prof)} / {len(ok_df)} successful runs have a profiler comparison; "
    f"profiler vs. event diff: median {prof_table['cuda_diff_pct'].median():+.1f}%, "
    f"range {prof_table['cuda_diff_pct'].min():+.1f}% to {prof_table['cuda_diff_pct'].max():+.1f}%"
)
show(prof_table)

154 / 154 successful runs have a profiler comparison; profiler vs. event diff: median +4.6%, range +0.0% to +55.3%
| model         | arch            | mode   |   frames |   bs |   event_latency_ms |   profiler_cuda_ms |   profiler_cpu_ms |   cuda_diff_pct |
|:--------------|:----------------|:-------|---------:|-----:|-------------------:|-------------------:|------------------:|----------------:|
| S3D           | S3D             | Train  |       16 |    1 |              31.46 |              48.86 |             48.51 |           55.33 |
| S3D           | S3D             | Train  |       16 |    2 |              58.22 |              69.41 |             69.07 |           19.23 |
| S3D           | S3D             | Train  |       16 |    4 |             109.85 |             121.07 |            120.72 |           10.21 |
| S3D           | S3D             | Train  |       16 |    8 |             214.41 |             225.69 |            225.34 |            5.26 |
| S3D           | S3D      